# Elevation pairs in elevation bins, with night-clustered errors and the bounce comparison (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-11-20  
**Last Modified:** 2026-11-21  
**Status:** In Progress  
**Keywords:** elevation, look-up table, LUT, bounce test, v-mode 1, DZ(1,4), Danish 1.3, Trim, optical state, bootstrap

## Description

Science visit pairs on one night, at most 5 min apart (non-consecutive allowed), with
azimuth and camera rotator angle each changing by less than 5 deg, the band unchanged, and
an elevation change of at least 2.5 deg. Only nights after the last hexapod look-up table (LUT) change,
day_obs >= 20251209, less the nights that ran a different LUT setting (the `thermal_focus`
list plus 20251214 and 20251216, found on this notebook's LUT page).

For each visit, the 22/12 deviation (22 degrees of freedom, DOF; 12 v-modes) is recovered
from the Danish 1.3 corner wavefront sensor (CWFS) Zernikes, and the commanded Trim is read
from `value_added`. The plotted quantity is **deviation minus Trim**, the optical state as
the bounce test computes it (no hexapod LUT), as 12 v-modes, 22 DOF and DZ(k,j) for
k = 1 at j = 4..19, 22..26 and k = 2, 3 at j = 4..8. DZ(k,j) is the sensitivity-matrix row S(k,j) times the 22 DOF, so it is the
12-mode projection, not the full measured DZ.

The change across each pair (later minus earlier visit) is fit against the elevation change
with a Huber line, over all pairs and in three bins of the pair's midpoint elevation. Slope
errors are night-clustered sandwich errors, which treat each night as one independent unit;
a short night bootstrap checks them. The bin slopes are then drawn as short segments
on the A-B-A bounce-test points (B minus the 70 deg reference) from `bounce_aba`
variant `d13-batoid-22_12`, the same 22/12 basis and batoid intrinsic, anchored on the bounce
points interpolated at each bin midpoint.

**Output:** `pairs_bins.parquet`, `slopes_bins.csv`, `slopes_by_del.csv` and `elevation_pairs_bins.pdf` in the
dated run directory.

**Based on:** `elevation_pairs_state_v2.ipynb`; bounce points from the `bounce_aba`
product; LUT epoch from `thermal_focus/thermal_focus/thermal_focus.md`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-11-20 | Aaron Roodman | Initial version |
| 2026-11-21 | Aaron Roodman | Night-clustered sandwich errors replace the bootstrap; drop 20251214 and 20251216 (other M2 LUT); slopes split by abs(dEl) |
| 2026-11-21 | Aaron Roodman | Bounce points from `bounce_aba` d13-batoid-22_12 (22/12, v1..v12); abs(dEl) >= 2.5 deg |
| 2026-11-22 | Aaron Roodman | DZ, v-mode and DOF pages separate; DZ(1,j) for all j and DZ(2,j), DZ(3,j) for j = 4..8 |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Optical State per Visit](#analysis)
6. [Fits and Bootstrap](#fits)
7. [Plots](#plots)

<a id='params'></a>
## Parameters

In [1]:
from rubinwork.common import study_run

DAY_OBS_MIN = 20251209        # last hexapod LUT change (thermal_focus.md)
EXTRA_LUT_NIGHTS = [20251214, 20251216]  # M2 LUT dz about +1,300 um above the other nights
MIN_DELTA_EL_DEG = 2.5        # minimum |elevation change| [deg]
MAX_DELTA_AZ_DEG = 5.0        # maximum |azimuth change| [deg], wrapped to +-180
MAX_DELTA_ROT_DEG = 5.0       # maximum |camera rotator angle change| [deg]
MAX_DT_MIN = 5.0              # maximum time between the two visit starts [min]
CONSECUTIVE_ONLY = False
CWFS_VARIANT = 'd13-blitz_rso1071'   # cwfs_science variant: Danish 1.3 blitz unpaired
DOF_SET, N_MODES = 'standard_22', 12  # the 22/12 scheme
DZ_J = [4, 5, 6, 7, 8]        # DZ(1,j) terms in the summary and linearity tables, Noll j
# Every DZ(k,j) fitted and plotted: k = 1 at all bounce j, k = 2 and 3 at j = 4..8.
DZ_KJ = ([(1, j) for j in [*range(4, 20), *range(22, 27)]]
         + [(k, j) for k in (2, 3) for j in range(4, 9)])
EL_BINS = [(25.0, 40.0), (40.0, 55.0), (55.0, 75.0)]  # pair midpoint elevation bins [deg]
BOUNCE_VARIANT = 'd13-batoid-22_12'  # bounce_aba variant: 22/12, batoid intrinsic
BOUNCE_REF_EL_DEG = 70.0      # the bounce A elevation, where its delta is 0 by definition
N_BOOT_CHECK = 200            # night resamples checking the clustered errors of v1 and DZ(1,4)
DEL_RANGES = [(2.5, 5.0), (5.0, 10.0), (10.0, 90.0)]  # abs(dEl) ranges for the linearity table [deg]
PLOT_DEL_MAX_DEG = 30.0       # plotted |elevation change| range [deg]
Y_QUANTILES = (0.001, 0.999)  # y range of each panel, quantiles of the plotted pairs
MEDIAN_BIN_DEG = 2.0          # running-median bin width in elevation change [deg]
MIN_PER_MEDIAN_BIN = 20       # fewest pairs for a median point

# Pinned date: reruns overwrite the 2026-11-21 run.
run = study_run('aos/elevation_pairs', 'bins_d13_22_12_dt5min_del2p5_lut20251209', date='20261121')

<a id='setup'></a>
## Setup & Imports

In [2]:
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import statsmodels.api as sm

from rubinwork import aos_state
from rubinwork.common.utils import nmad, repo_root, setup_plotting
from rubinwork.products import cwfs_science
from rubinwork.products.bounce_aba import reader as bounce_aba
from rubinwork.products import value_added as efd_db
from rubinwork.thermal_focus import lib as tf_lib

CODE_DIR = repo_root() / 'aos' / 'elevation_pairs' / 'code'
sys.path.insert(0, str(CODE_DIR))
import pairs_lib as PL

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
setup_plotting()

<a id='functions'></a>
## Helper Functions

Pair search, state recovery and the bootstrap engine are in `code/pairs_lib.py`.

In [3]:
DOF_IDX = aos_state.DOF_SETS[DOF_SET]
DOF_LABEL = {0: 'M2 dz', 1: 'M2 dx', 2: 'M2 dy', 3: 'M2 rx', 4: 'M2 ry',
             5: 'Cam dz', 6: 'Cam dx', 7: 'Cam dy', 8: 'Cam rx', 9: 'Cam ry',
             **{k: f'M1M3 B{k - 9}' for k in range(10, 30)},
             **{k: f'M2 B{k - 29}' for k in range(30, 50)}}
#: Hexapod tilts are stored in deg; every other DOF in um.
DOF_UNIT = {k: ('deg' if k in (3, 4, 8, 9) else 'um') for k in range(50)}
LUT_OFFSET_NIGHTS = sorted([n for n in tf_lib.LUT_EPOCH_OFFSET_NIGHTS if n >= DAY_OBS_MIN]
                           + EXTRA_LUT_NIGHTS)

# (quantity, delta column, label, unit), in page order: DZ, v-modes, DOF.
QUANT = ([(f'dz{k}_{j}', f'd_dz{k}_{j}', f'DZ({k},{j})', 'um of wavefront') for k, j in DZ_KJ]
         + [(f'v{m}', f'd_dmt_v{m}', f'v{m}', 'dimensionless') for m in range(1, N_MODES + 1)]
         + [(f'dof{k}', f'd_dmt_dof{k}', f'dof{k} {DOF_LABEL[k]}', DOF_UNIT[k]) for k in DOF_IDX])
QLAB = {q: (lab, u) for q, _, lab, u in QUANT}
GROUPS = {'DZ(1,j)': [f'dz{k}_{j}' for k, j in DZ_KJ if k == 1],
          'DZ(2,j) and DZ(3,j)': [f'dz{k}_{j}' for k, j in DZ_KJ if k > 1],
          'v-modes': [f'v{m}' for m in range(1, N_MODES + 1)],
          '22 DOF': [f'dof{k}' for k in DOF_IDX]}


def huber_fit(x, y):
    # statsmodels Huber line; returns slope, formal error, scale and the correlations.
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    res = sm.RLM(y, sm.add_constant(x), M=sm.robust.norms.HuberT()).fit()
    return dict(slope=res.params[1], slope_err=res.bse[1], intercept=res.params[0],
                intercept_err=res.bse[0], scale=res.scale, r=stats.pearsonr(x, y)[0],
                rho=stats.spearmanr(x, y)[0], n=len(x), nmad_resid=nmad(y - res.fittedvalues))


def short_unit(unit):
    return '' if unit == 'dimensionless' else unit.replace(' of wavefront', '')

<a id='data'></a>
## Data Access

Science visits with pointing and band from ConsDB, the Trim (`dof0`..`dof49`) and hexapod LUT from `value_added`, the Danish 1.3 per-CCD Zernikes from `cwfs_science`, and the A-B-A bounce points from `bounce_aba`.

In [4]:
trim_cols = [f'dof{k}' for k in range(50)]
lut_cols = [f'lut_dof{k}' for k in range(10)]
vis = efd_db.visits(columns=['visit_id', 'day_obs', 'seq_num'] + trim_cols + lut_cols)
vis = efd_db.join_consdb(vis, groups=('meta',))
sci_all = vis[vis['img_type'] == 'science'].dropna(
    subset=['band', 'obs_start_mjd', 'altitude_deg', 'azimuth_deg_consdb', 'rotator_angle_deg'])
keep = (sci_all['day_obs'] >= DAY_OBS_MIN) & ~sci_all['day_obs'].isin(LUT_OFFSET_NIGHTS)
sci = sci_all[keep]
print(f'{len(sci)} science visits on {sci["day_obs"].nunique()} nights from day_obs {DAY_OBS_MIN}, '
      f'less LUT-offset nights {LUT_OFFSET_NIGHTS}')
pairs = PL.find_pairs(sci, MIN_DELTA_EL_DEG, MAX_DELTA_AZ_DEG, MAX_DELTA_ROT_DEG,
                      MAX_DT_MIN, CONSECUTIVE_ONLY)
print(f'{len(pairs)} pairs over {pairs["day_obs"].nunique()} nights')

ccd = cwfs_science.load_table('ccd_visits', CWFS_VARIANT)
pair_visits = np.unique(np.r_[pairs['visit_id_1'], pairs['visit_id_2']])
z_dev = PL.corner_deviation(ccd, pair_visits)
print(f'{len(z_dev)} of {len(pair_visits)} pair visits have a complete {CWFS_VARIANT} corner '
      f'deviation; {CWFS_VARIANT} spans day_obs {ccd["day_obs"].min()} to {ccd["day_obs"].max()}')
# DZ is the 22/12 subspace fit, to match the pairs' S(1,j) x DOF.
b_state = bounce_aba.load_table('aba_state', BOUNCE_VARIANT)
b_state['quantity'] = np.where(b_state['kind'] == 'vmode', 'v' + b_state['index'].astype(str),
                               'dof' + b_state['index'].astype(str))
b_dz = bounce_aba.load_table('aba_dz', BOUNCE_VARIANT)
b_dz = b_dz[[kj in DZ_KJ for kj in zip(b_dz['k'], b_dz['j'])]].copy()
b_dz['quantity'] = 'dz' + b_dz['k'].astype(str) + '_' + b_dz['j'].astype(str)
b_dz = b_dz.drop(columns=['delta', 'err']).rename(columns={'fit': 'delta', 'fit_err': 'err'})
bcols = ['night', 'leg', 'b_elev_deg', 'quantity', 'delta', 'err']
bounce = pd.concat([b_state[bcols], b_dz[bcols]], ignore_index=True)
print(f'{len(bounce)} bounce points, {bounce["quantity"].nunique()} quantities, '
      f'B elevations {sorted(bounce["b_elev_deg"].unique())} deg')

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead.


        Use pytest instead. [astropy.utils.decorators]
        Use pytest instead.


72669 science visits on 138 nights from day_obs 20251209, less LUT-offset nights [20251210, 20251211, 20251212, 20251214, 20251216, 20251218, 20251219, 20260115, 20260116]


75248 pairs over 121 nights


42428 of 48338 pair visits have a complete d13-blitz_rso1071 corner deviation; d13-blitz_rso1071 spans day_obs 20251021 to 20260714
455 bounce points, 65 quantities, B elevations [np.float64(30.0), np.float64(40.0), np.float64(50.0), np.float64(60.0), np.float64(75.0)] deg


<a id='analysis'></a>
## Optical State per Visit

22/12 recovery with `aos_state.recover_optical_state`. `pairs_lib` stores state = Trim - deviation; deviation - Trim is its negative. DZ(k,j) = S(k,j) times the 22 deviation - Trim DOF, with S the v13 sensitivity matrix (`ofc_data.sensitivity_matrix`, indexed by k and Noll j).

In [5]:
se = aos_state.make_state_estimator(dof_set=DOF_SET, n_modes=N_MODES)
per_visit = PL.recover_states(z_dev, sci.set_index('visit_id'), se, N_MODES, DOF_SET)
print(f'{len(per_visit)} visits with a state and a Trim; median subspace residual '
      f'{per_visit["resid_rms_um"].median():.4f} um of wavefront')
S = np.asarray(se.ofc_data.sensitivity_matrix)
dmt = -per_visit[[f'state_dof{k}' for k in DOF_IDX]].to_numpy()
for m in range(1, N_MODES + 1):
    per_visit[f'dmt_v{m}'] = -per_visit[f'state_v{m}']
for k in DOF_IDX:
    per_visit[f'dmt_dof{k}'] = -per_visit[f'state_dof{k}']
for k, j in DZ_KJ:
    per_visit[f'dz{k}_{j}'] = dmt @ S[k, j, DOF_IDX]
v1_dz14 = se.get_dofs_from_vmodes(np.r_[1.0, np.zeros(N_MODES - 1)]) @ S[1, 4, DOF_IDX]
print(f'S(1,4) check: dof0 {S[1, 4, 0]:+.4e}, dof5 {S[1, 4, 5]:+.4e} um of wavefront per um; '
      f'DZ(1,4) per unit v1 = {v1_dz14:+.3f} um of wavefront')

names = ([f'dmt_v{m}' for m in range(1, N_MODES + 1)] + [f'dmt_dof{k}' for k in DOF_IDX]
         + [f'dz{k}_{j}' for k, j in DZ_KJ] + ['resid_rms_um'])
pairs = PL.attach_deltas(pairs, per_visit, names)
ps = pairs[pairs['d_dmt_v1'].notna()].reset_index(drop=True)
ps['el_mid_deg'] = ps['el_1_deg'] + 0.5 * ps['d_el_deg']
n_vis = len(np.unique(np.r_[ps['visit_id_1'], ps['visit_id_2']]))
print(f'{len(ps)} of {len(pairs)} pairs have a state at both visits, over '
      f'{ps["day_obs"].nunique()} nights and {n_vis} distinct visits')
print('|dEl| distribution [deg]:', np.histogram(ps['d_el_deg'].abs(), [0, 1, 2, 5, 10, 15, 20, 30, 90])[0])

42428 visits with a state and a Trim; median subspace residual 0.0950 um of wavefront
S(1,4) check: dof0 -1.5341e-02, dof5 -1.5914e-02 um of wavefront per um; DZ(1,4) per unit v1 = +17.418 um of wavefront


66306 of 75248 pairs have a state at both visits, over 107 nights and 42321 distinct visits
|dEl| distribution [deg]: [    0     0 37123 25259  3424   402    93     5]


<a id='fits'></a>
## Fits and Errors

Point fits by statsmodels Huber. Errors are night-clustered sandwich errors (`PL.huber_cluster_slope_err`). A night bootstrap of `N_BOOT_CHECK` resamples (`PL.night_bootstrap`) checks them for v1 and DZ(1,4). The linearity table refits v1 and DZ(1,4..8) in abs(dEl) ranges.

In [6]:
samples = {'all': np.ones(len(ps), bool)}
for lo, hi in EL_BINS:
    samples[f'{lo:g}-{hi:g}'] = (ps['el_mid_deg'] >= lo) & (ps['el_mid_deg'] < hi)
samples = {k: np.asarray(v) for k, v in samples.items()}
x = ps['d_el_deg'].to_numpy()
Y = ps[[col for _, col, _, _ in QUANT]].to_numpy()
rows, scale, start = [], {}, {}
for s, m in samples.items():
    fits = [huber_fit(x[m], Y[m, i]) for i in range(len(QUANT))]
    scale[s] = np.array([f['scale'] for f in fits])
    start[s] = np.array([[f['intercept'] for f in fits], [f['slope'] for f in fits]])
    rows += [dict(quantity=q, sample=s, unit=f'{u}/deg', **f) for (q, _, _, u), f in zip(QUANT, fits)]
m10 = np.abs(x) >= 10
rows += [dict(quantity=q, sample='abs_del_ge_10', unit=f'{u}/deg', **huber_fit(x[m10], Y[m10, i]))
         for i, (q, _, _, u) in enumerate(QUANT)]
slopes = pd.DataFrame(rows)

nights_all = ps['day_obs'].to_numpy()
t0 = time.time()
for s_, m in samples.items():
    sel = slopes['sample'] == s_
    slopes.loc[sel, 'slope_err_cl'] = PL.huber_cluster_slope_err(
        x[m], Y[m], nights_all[m], start[s_], scale[s_])
t_cl = time.time() - t0
slopes['cl_over_formal'] = slopes['slope_err_cl'] / slopes['slope_err']
slopes['signif'] = slopes['slope'] / slopes['slope_err_cl']
print(f'night-clustered errors: {len(samples)} samples x {len(QUANT)} quantities in {t_cl:.2f} s')

# Check: night bootstrap of v1 and DZ(1,4) with the fixed-scale engine.
qi = [[q for q, _, _, _ in QUANT].index(q) for q in ('v1', 'dz1_4')]
t0 = time.time()
chk = PL.night_bootstrap(x, Y[:, qi], nights_all, samples,
                         {k: v[qi] for k, v in scale.items()},
                         {k: v[:, qi] for k, v in start.items()}, n_boot=N_BOOT_CHECK)
t_chk = time.time() - t0
boot_check = []
for s_ in samples:
    for j, q in enumerate(('v1', 'dz1_4')):
        f = slopes[(slopes['quantity'] == q) & (slopes['sample'] == s_)].iloc[0]
        boot_check.append((s_, q, f['slope_err_cl'], chk[s_][:, j].std(ddof=1)))
        print(f'{s_:>6} {q:<6} slope error: clustered {boot_check[-1][2]:.3e}, bootstrap '
              f'{boot_check[-1][3]:.3e} per deg ({QLAB[q][1]})')
print(f'{N_BOOT_CHECK}-resample bootstrap check took {t_chk:.0f} s')

# Linearity: slopes by abs(dEl) range inside each sample.
lin = []
lq = ['v1'] + [f'dz1_{j}' for j in DZ_J]
li = [[q for q, _, _, _ in QUANT].index(q) for q in lq]
for s_, m in samples.items():
    for lo, hi in DEL_RANGES:
        mm = m & (np.abs(x) >= lo) & (np.abs(x) < hi)
        fits = [huber_fit(x[mm], Y[mm, i]) for i in li]
        par = np.array([[f['intercept'] for f in fits], [f['slope'] for f in fits]])
        err = PL.huber_cluster_slope_err(x[mm], Y[mm][:, li], nights_all[mm], par,
                                         np.array([f['scale'] for f in fits]))
        lin += [dict(sample=s_, del_lo=lo, del_hi=hi, quantity=q, slope=f['slope'],
                     slope_err_cl=e, n=f['n']) for q, f, e in zip(lq, fits, err)]
lin = pd.DataFrame(lin)
cols = ['quantity', 'sample', 'unit', 'slope', 'slope_err', 'slope_err_cl', 'cl_over_formal',
        'signif', 'intercept', 'r', 'rho', 'n']
print(slopes[slopes['quantity'].isin(['v1'] + [f'dz1_{j}' for j in DZ_J])][cols]
      .to_string(index=False, float_format=lambda v: f'{v:.4g}'))

night-clustered errors: 4 samples x 65 quantities in 0.29 s


   all v1     slope error: clustered 5.032e-05, bootstrap 5.101e-05 per deg (dimensionless)
   all dz1_4  slope error: clustered 7.499e-04, bootstrap 7.552e-04 per deg (um of wavefront)
 25-40 v1     slope error: clustered 1.974e-04, bootstrap 1.860e-04 per deg (dimensionless)
 25-40 dz1_4  slope error: clustered 3.212e-03, bootstrap 3.181e-03 per deg (um of wavefront)
 40-55 v1     slope error: clustered 7.476e-05, bootstrap 7.542e-05 per deg (dimensionless)
 40-55 dz1_4  slope error: clustered 1.079e-03, bootstrap 1.065e-03 per deg (um of wavefront)
 55-75 v1     slope error: clustered 4.624e-05, bootstrap 4.540e-05 per deg (dimensionless)
 55-75 dz1_4  slope error: clustered 6.705e-04, bootstrap 6.755e-04 per deg (um of wavefront)
200-resample bootstrap check took 27 s


quantity        sample                unit      slope  slope_err  slope_err_cl  cl_over_formal  signif  intercept         r       rho     n
   dz1_4           all um of wavefront/deg  0.0003588  0.0001217     0.0007499           6.163  0.4785   0.001661  0.004706  0.009612 66306
   dz1_5           all um of wavefront/deg  0.0008242  0.0001826     0.0009654           5.286  0.8537  0.0004134   0.02356   0.01581 66306
   dz1_6           all um of wavefront/deg  0.0001316  0.0001925       0.00105           5.454  0.1253  -0.002103 -0.007142  0.002618 66306
   dz1_7           all um of wavefront/deg  -0.003903  8.126e-05     0.0004062           4.998  -9.609  -0.007985   -0.1037   -0.1552 66306
   dz1_8           all um of wavefront/deg  -0.001575  7.037e-05     0.0002555            3.63  -6.165  -0.004278  -0.04012  -0.07942 66306
      v1           all   dimensionless/deg  0.0001151  1.009e-05     5.032e-05           4.989   2.287  0.0001018   0.03352   0.03804 66306
   dz1_4         25-

In [7]:
pairs.to_parquet(run.path / 'pairs_bins.parquet', index=False)
slopes.to_csv(run.path / 'slopes_bins.csv', index=False, float_format='%.6g')
lin.to_csv(run.path / 'slopes_by_del.csv', index=False, float_format='%.6g')

<a id='plots'></a>
## Plots

One PDF: summary, slope table, LUT check, bounce comparison (DZ(1,j), DZ(2,j) and DZ(3,j), v1..v12 and the 22 DOF, each group on its own pages), then one page per quantity with all pairs and the three midpoint-elevation bins.

In [8]:
WHAT = ('Plotted: change in deviation - Trim (the bounce-test optical state), later minus '
        'earlier visit; no hexapod LUT.')
SAMPLE_TITLE = {'all': 'all pairs', **{f'{lo:g}-{hi:g}': f'midpoint elevation {lo:g}-{hi:g} deg'
                                       for lo, hi in EL_BINS}}


def fit_of(q, s):
    return slopes[(slopes['quantity'] == q) & (slopes['sample'] == s)].iloc[0]


def hexbin_panel(ax, fig, xv, yv, f, ylabel, title):
    # x is the elevation change [deg]; y range spans Y_QUANTILES of the plotted pairs.
    ok = np.isfinite(xv) & np.isfinite(yv) & (np.abs(xv) <= PLOT_DEL_MAX_DEG)
    xv, yv = xv[ok], yv[ok]
    ylo, yhi = np.quantile(yv, Y_QUANTILES)
    pad = 0.05 * (yhi - ylo)
    ylo, yhi = ylo - pad, yhi + pad
    hb = ax.hexbin(xv, yv, gridsize=(60, 30), mincnt=1, bins='log', cmap='viridis',
                   extent=(-PLOT_DEL_MAX_DEG, PLOT_DEL_MAX_DEG, ylo, yhi))
    fig.colorbar(hb, ax=ax, label='pairs per bin (log scale)')
    edges = np.arange(-PLOT_DEL_MAX_DEG, PLOT_DEL_MAX_DEG + MEDIAN_BIN_DEG / 2, MEDIAN_BIN_DEG)
    idx = np.digitize(xv, edges)
    xm, ym, em = [], [], []
    for i in np.unique(idx):
        yy = yv[idx == i]
        if len(yy) >= MIN_PER_MEDIAN_BIN:
            xm.append(np.median(xv[idx == i]))
            ym.append(np.median(yy))
            # Error of a median: 1.253 sigma / sqrt(n), with sigma from nMAD.
            em.append(1.253 * nmad(yy) / np.sqrt(len(yy)))
    ax.errorbar(xm, ym, yerr=em, fmt='o', color='white', mec='k', ms=4, ecolor='k',
                capsize=2, label=f'median per {MEDIAN_BIN_DEG:g} deg bin')
    xx = np.array([-PLOT_DEL_MAX_DEG, PLOT_DEL_MAX_DEG])
    ax.plot(xx, f['intercept'] + f['slope'] * xx, color='C3', lw=1.5, label='Huber fit')
    ax.axhline(0, color='0.6', lw=0.5)
    ax.axvline(0, color='0.6', lw=0.5)
    ax.set_xlim(-PLOT_DEL_MAX_DEG, PLOT_DEL_MAX_DEG)
    ax.set_ylim(ylo, yhi)
    ax.set_xlabel('elevation change, later minus earlier [deg]')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7, loc='upper left')


def fit_title(head, f, unit):
    u = short_unit(unit)
    return (f'{head}\nslope {f["slope"]:+.3g} +- {f["slope_err_cl"]:.2g} {u}/deg '
            f'({f["signif"]:+.1f} sigma, night-clustered)\n'
            f'Pearson r = {f["r"]:+.3f}, Spearman rho = {f["rho"]:+.3f}, n = {f["n"]} pairs')


def bounce_curve(q):
    # Bounce points per B elevation (40 deg = mean of its nights), plus 0 at the reference.
    b = bounce[bounce['quantity'] == q].groupby('b_elev_deg')['delta'].mean()
    b.loc[BOUNCE_REF_EL_DEG] = 0.0
    return b.sort_index()


def bounce_panel(ax, q):
    lab, unit = QLAB[q]
    b = bounce[bounce['quantity'] == q]
    for (night, el), g in b.groupby(['night', 'b_elev_deg']):
        off = 0.0
        if el == 40.0:
            off = {20260418: -0.8, 20260419: 0.0, 20260513: 0.8}.get(night, 0.0)
        ax.errorbar(el + off, g['delta'].iloc[0], yerr=g['err'].iloc[0], fmt='o', color='k',
                    ms=5, capsize=2)
    ax.plot([BOUNCE_REF_EL_DEG], [0.0], 'o', mfc='none', mec='k', ms=7)
    curve = bounce_curve(q)
    ax.plot(curve.index, curve.values, color='0.6', lw=0.8, ls=':', zorder=0)
    for i, (lo, hi) in enumerate(EL_BINS):
        f = fit_of(q, f'{lo:g}-{hi:g}')
        mid = 0.5 * (lo + hi)
        y0 = np.interp(mid, curve.index, curve.values)
        xx = np.linspace(lo, hi, 20)
        col = f'C{i}'
        ax.fill_between(xx, y0 + (f['slope'] - f['slope_err_cl']) * (xx - mid),
                        y0 + (f['slope'] + f['slope_err_cl']) * (xx - mid), color=col, alpha=0.25)
        ax.plot(xx, y0 + f['slope'] * (xx - mid), color=col, lw=2,
                label=f'{lo:g}-{hi:g} deg: {f["slope"]:+.3g} +- {f["slope_err_cl"]:.2g} '
                      f'{short_unit(unit)}/deg')
    ax.axhline(0, color='0.6', lw=0.5)
    ax.set_xlabel('elevation [deg] (bounce: B elevation; A = 70 deg, open circle)')
    ax.set_ylabel(f'{lab} [{unit}]')
    ax.set_title(f'{lab}: bounce B - A (points) and elevation-pair slopes (segments)', fontsize=9)
    ax.legend(fontsize=7)


with PdfPages(run.path / 'elevation_pairs_bins.pdf') as pdf:
    lines = [f'Elevation pairs in midpoint-elevation bins: 22/12 from {CWFS_VARIANT}', '',
             'PLOTTED QUANTITY: deviation - Trim, the optical state as the bounce test computes',
             'it (no hexapod LUT). Change = later visit minus earlier visit.',
             'DZ(k,j) = S(k,j) x 22 DOF: the 12-mode projection, not the full measured DZ.', '',
             f'Nights: day_obs >= {DAY_OBS_MIN} (last LUT change), less {LUT_OFFSET_NIGHTS}',
             f'Cuts: |dEl| >= {MIN_DELTA_EL_DEG:g} deg, |dAz| < {MAX_DELTA_AZ_DEG:g} deg, |dRot| < {MAX_DELTA_ROT_DEG:g} deg, '
             f'same band, dt <= {MAX_DT_MIN:g} min, non-consecutive allowed',
             f'Pairs: {len(pairs)} found, {len(ps)} with a state at both visits, '
             f'{ps["day_obs"].nunique()} nights, {n_vis} distinct visits',
             'Bins by pair midpoint elevation (El1 + El2) / 2: '
             + ', '.join(f'{k} deg n = {int(v.sum())}' for k, v in samples.items() if k != 'all'),
             '',
             f'Errors: night-clustered sandwich over {ps["day_obs"].nunique()} nights ({t_cl:.1f} s).',
             f'  Check against a {N_BOOT_CHECK}-resample night bootstrap ({t_chk:.0f} s), slope error per deg:']
    lines += [f'    {s_:>6} {q:<6} clustered {a:.3e}, bootstrap {b:.3e} ({QLAB[q][1]})'
              for s_, q, a, b in boot_check]
    lines += ['', 'Huber slope per deg of elevation +- clustered error (clustered/formal error ratio,',
              'dimensionless). v1 dimensionless; DZ in um of wavefront.', '']
    head = f'  {"":<9}' + ''.join(f'{s:>27}' for s in samples) + f'{"|dEl|>=10, formal":>24}'
    lines.append(head)
    for q in ['v1'] + [f'dz1_{j}' for j in DZ_J]:
        txt = f'  {QLAB[q][0]:<9}'
        for s in samples:
            f = fit_of(q, s)
            txt += f'{f["slope"]:+11.3e} +-{f["slope_err_cl"]:8.1e} ({f["cl_over_formal"]:3.1f})'
        f = fit_of(q, 'abs_del_ge_10')
        txt += f'  {f["slope"]:+10.3e} +-{f["slope_err"]:7.1e}'
        lines.append(txt)
    fig = plt.figure(figsize=(11, 8.5))
    fig.text(0.03, 0.97, '\n'.join(lines), fontsize=7.5, va='top', family='monospace')
    pdf.savefig(fig)
    plt.close(fig)

    lines = ['Huber slope per deg of elevation of deviation - Trim +- night-clustered error',
             '(units: v-modes 1/deg; DZ um/deg; DOF um/deg, tilts deg/deg)', '',
             f'  {"quantity":<16}' + ''.join(f'{s:>26}' for s in samples)]
    for q, _, lab, u in QUANT:
        txt = f'  {lab:<16}'
        for s in samples:
            f = fit_of(q, s)
            txt += f'{f["slope"]:+11.3e} +-{f["slope_err_cl"]:8.1e} {f["signif"]:+5.1f}s'
        lines.append(txt)
    lines += ['', 's = slope / clustered error (dimensionless)']
    fig = plt.figure(figsize=(11, 8.5))
    fig.text(0.03, 0.97, '\n'.join(lines), fontsize=7.5, va='top', family='monospace')
    pdf.savefig(fig)
    plt.close(fig)

    lines = ['Linearity: Huber slope per deg of elevation +- night-clustered error, refit in',
             'ranges of abs(dEl) inside each sample. v1 dimensionless; DZ in um of wavefront.', '']
    for s_ in samples:
        lines.append(f'{SAMPLE_TITLE[s_]}')
        lines.append(f'  {"":<9}' + ''.join(f'{f"|dEl| {lo:g}-{hi:g} deg":>28}' for lo, hi in DEL_RANGES))
        for q in lq:
            txt = f'  {QLAB[q][0]:<9}'
            for lo, hi in DEL_RANGES:
                r_ = lin[(lin['sample'] == s_) & (lin['quantity'] == q) & (lin['del_lo'] == lo)].iloc[0]
                txt += f'{r_["slope"]:+11.3e} +-{r_["slope_err_cl"]:8.1e} n={r_["n"]:<6d}'
            lines.append(txt)
        lines.append('')
    fig = plt.figure(figsize=(11, 8.5))
    fig.text(0.03, 0.97, '\n'.join(lines), fontsize=7.5, va='top', family='monospace')
    pdf.savefig(fig)
    plt.close(fig)

    fig, axs = plt.subplots(1, 2, figsize=(14, 6), layout='constrained')
    lv = sci_all[sci_all['day_obs'] >= 20251021]
    after = (lv['day_obs'] >= DAY_OBS_MIN) & ~lv['day_obs'].isin(LUT_OFFSET_NIGHTS)
    for ax, k in zip(axs, (5, 0)):
        ax.scatter(lv.loc[~after, 'altitude_deg'], lv.loc[~after, f'lut_dof{k}'], s=2, c='0.75',
                   label=f'before {DAY_OBS_MIN} or LUT-offset night')
        for ib, b_ in enumerate('ugrizy'):
            mb = after & (lv['band'] == b_)
            ax.scatter(lv.loc[mb, 'altitude_deg'], lv.loc[mb, f'lut_dof{k}'], s=2, c=f'C{ib}',
                       label=f'band {b_}')
        ax.set_xlabel('elevation [deg]')
        ax.set_ylabel(f'hexapod LUT {DOF_LABEL[k]} [um]')
        ax.set_title(f'LUT {DOF_LABEL[k]} against elevation; colour: band, used nights only', fontsize=9)
        ax.legend(fontsize=7, markerscale=4)
    fig.suptitle('Hexapod LUT check on the nights used: the camera dz branches are per-band focus '
                 'offsets; pairs keep the band fixed. Spread along a branch is the rotator-angle term.',
                 fontsize=10)
    pdf.savefig(fig)
    plt.close(fig)

    for gname, bq in GROUPS.items():
        for p0 in range(0, len(bq), 4):
            fig, axs = plt.subplots(2, 2, figsize=(14, 10.5), layout='constrained')
            for ax, q in zip(axs.flat, bq[p0:p0 + 4]):
                bounce_panel(ax, q)
            for ax in list(axs.flat)[len(bq[p0:p0 + 4]):]:
                ax.axis('off')
            fig.suptitle(f'Bounce comparison, {gname}: A-B-A delta (B - A, A = 70 deg) against B elevation; '
                         f'bounce {BOUNCE_VARIANT} (22/12)\n'
                         'segments = deviation - Trim slope per midpoint bin, +-1 sigma (night-clustered), '
                         'anchored on the bounce points interpolated at the bin midpoint',
                         fontsize=10)
            pdf.savefig(fig)
            plt.close(fig)

    for q, col, lab, unit in QUANT:
        fig, axs = plt.subplots(2, 2, figsize=(14, 10.5), layout='constrained')
        for ax, (s, m) in zip(axs.flat, samples.items()):
            f = fit_of(q, s)
            hexbin_panel(ax, fig, x[m], ps.loc[m, col].to_numpy(), f,
                         f'{lab} change, deviation - Trim [{unit}]',
                         fit_title(f'{lab}, {SAMPLE_TITLE[s]}', f, unit))
        fig.suptitle(f'{lab}: change against elevation change. {WHAT}', fontsize=9)
        pdf.savefig(fig)
        plt.close(fig)
print(f'Wrote {run.path}')

Wrote /sdf/group/rubin/u/roodman/LSST/rubin-work/studies/aos/elevation_pairs/20261121_bins_d13_22_12_dt5min_del2p5_lut20251209
